# Berlin EMS Demand Predictor – Full Notebook

ML Zoomcamp 2026 Midterm Project

This notebook contains:
1. Data loading & cleaning
2. Exploratory Data Analysis
3. Feature engineering
4. Model training, comparison and hyper-parameter tuning
5. Final model selection

The production training script is `train.py`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.data import load_raw_data
from src.features import prepare_features, add_calendar_features
from src.models import get_models, evaluate_model, time_series_cv_score

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1. Load Data

In [ ]:
df = load_raw_data()
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

## 2. Exploratory Data Analysis

In [ ]:
# Time series of target
plt.figure(figsize=(14, 4))
plt.plot(df["mission_created_date"], df["mission_count_ems"])
plt.title("Daily EMS Missions in Berlin")
plt.ylabel("EMS missions")
plt.tight_layout()
plt.show()

In [ ]:
df_cal = add_calendar_features(df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_cal.groupby("dayofweek")["mission_count_ems"].mean().plot(kind="bar", ax=axes[0], title="Avg by Day of Week")
df_cal.groupby("month")["mission_count_ems"].mean().plot(kind="bar", ax=axes[1], title="Avg by Month")
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
X, y, df_enriched = prepare_features(df)
print(X.shape)
X.head()

## 4. Model Training & Comparison

In [ ]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

models = get_models()
results = {}

for name, model in models.items():
    print(f"Training {name}...")
    metrics = evaluate_model(model, X_train, y_train, X_test, y_test)
    metrics["cv_rmse"] = time_series_cv_score(model, X_train, y_train)
    results[name] = metrics
    print(f"  RMSE={metrics['rmse']:.1f}  MAE={metrics['mae']:.1f}  R2={metrics['r2']:.3f}  CV_RMSE={metrics['cv_rmse']:.1f}")

In [ ]:
pd.DataFrame(results).T.sort_values("rmse")

## 5. Final Model

The best model (lowest hold-out RMSE) is selected and saved by `train.py`.